# Software-Engineered Chest X-Ray Analysis and Explanation System

**Notebook:** Platform Readiness and Environment Validation  
**Subject:** AIMLCZG546 - Software Engineering for Machine Learning  
**Assessment:** Assignment II  
**Group:** 89

## Group Details & Contribution

| BITS ID | Name | Engineering contribution | Weight |
|---|---|---|---:|
| 2024ac05672@wilp.bits-pilani.ac.in | Ananda Vadivel D | Requirements formulation, GR4ML view preparation, quality requirement justification | 25% |
| 2024ac05968@wilp.bits-pilani.ac.in | Adurti Sai Venkatesh | Dataset preparation, exploratory data analysis, visualization support, data quality validation | 25% |
| 2024ac05653@wilp.bits-pilani.ac.in | D Mallikarjuna Reddy | ML pipeline implementation, model comparison, final evaluation, report integration | 25% |
| 2024ac05055@wilp.bits-pilani.ac.in | Krupashankar Subramani | System architecture, architectural pattern mapping, model registry artifacts, inference and application validation | 25% |

## Notebook Index

1. [Platform Readiness and Environment Validation](#platform-readiness-and-environment-validation)
2. [1.1 Runtime and Storage Identification](#11-runtime-and-storage-identification)
3. [1.2 Solution Storage Initialization](#12-solution-storage-initialization)
4. [1.3 Solution Workspace Structure](#13-solution-workspace-structure)
5. [1.4 Data and Artifact Storage Structure](#14-data-and-artifact-storage-structure)
6. [2. Compute Resource Validation](#2-compute-resource-validation)
7. [2.1 CPU and Memory Allocation](#21-cpu-and-memory-allocation)
8. [2.2 GPU Hardware Allocation](#22-gpu-hardware-allocation)
9. [2.3 PyTorch and CUDA Integration](#23-pytorch-and-cuda-integration)
10. [3. Software Dependency Validation](#3-software-dependency-validation)
11. [3.1 Solution Package Inventory](#31-solution-package-inventory)
12. [3.3 Dependency Import Validation](#33-dependency-import-validation)
13. [3.4 Kubeflow Dependency Compatibility](#34-kubeflow-dependency-compatibility)
14. [4. Functional Runtime Validation](#4-functional-runtime-validation)
15. [4.1 GPU Tensor and Mixed-Precision Test](#41-gpu-tensor-and-mixed-precision-test)
16. [5. Centralized Path Configuration](#5-centralized-path-configuration)
17. [5.1 Solution Path Registry](#51-solution-path-registry)
18. [5.2 Runtime Cache and Experiment Tracking Configuration](#52-runtime-cache-and-experiment-tracking-configuration)
19. [6. Environment Readiness Assessment](#6-environment-readiness-assessment)
20. [6.1 Readiness Gate](#61-readiness-gate)
21. [Readiness Outcome](#readiness-outcome)

---

The notebook records the executable implementation, retained runtime outputs, and verifiable engineering evidence for this component.


# Platform Readiness and Environment Validation

This notebook validates the runtime, compute resources, storage configuration and software dependencies required for the Software-Engineered Chest X-Ray Analysis and Explanation System.

The validation establishes a reliable foundation for data processing, multilabel model training, visual explainability, API services, language-model integration and operational monitoring.


## 1.1 Runtime and Storage Identification

This section identifies the active Python runtime, notebook location and available storage volumes. The results will be used to separate lightweight solution code from storage-intensive datasets, model artifacts and experiment outputs.


In [14]:
import os
import platform
import shutil
import sys
from pathlib import Path

NOTEBOOK_DIRECTORY = Path.cwd()
SERVER_ROOT = Path.home()
DATA_VOLUME = SERVER_ROOT / "apicdsa2-datavol-1"

print("RUNTIME DETAILS")
print("-" * 100)
print(f"{'Python version':<28}: {platform.python_version()}")
print(f"{'Python executable':<28}: {sys.executable}")
print(f"{'Operating system':<28}: {platform.platform()}")
print(f"{'Notebook directory':<28}: {NOTEBOOK_DIRECTORY}")
print(f"{'Jupyter server root':<28}: {SERVER_ROOT}")
print()

print("STORAGE DETAILS")
print("-" * 100)

paths_to_check = {
    "Workspace volume": SERVER_ROOT,
    "Dedicated data volume": DATA_VOLUME,
}

for name, path in paths_to_check.items():
    if path.exists():
        usage = shutil.disk_usage(path)

        print(f"{name}")
        print(f"  Path       : {path}")
        print(f"  Writable   : {os.access(path, os.W_OK)}")
        print(f"  Total      : {usage.total / (1024 ** 3):.2f} GiB")
        print(f"  Used       : {usage.used / (1024 ** 3):.2f} GiB")
        print(f"  Available  : {usage.free / (1024 ** 3):.2f} GiB")
    else:
        print(f"{name}")
        print(f"  Path       : {path}")
        print("  Status     : Not found")

    print()

RUNTIME DETAILS
----------------------------------------------------------------------------------------------------
Python version              : 3.11.11
Python executable           : /opt/conda/bin/python
Operating system            : Linux-6.8.0-117-generic-x86_64-with-glibc2.35
Notebook directory          : /home/jovyan
Jupyter server root         : /home/jovyan

STORAGE DETAILS
----------------------------------------------------------------------------------------------------
Workspace volume
  Path       : /home/jovyan
  Writable   : True
  Total      : 7.78 GiB
  Used       : 0.00 GiB
  Available  : 7.76 GiB

Dedicated data volume
  Path       : /home/jovyan/apicdsa2-datavol-1
  Writable   : True
  Total      : 23.46 GiB
  Used       : 2.77 GiB
  Available  : 20.67 GiB



## 1.2 Solution Storage Initialization

This section establishes separate root directories for solution code and storage-intensive artifacts. The separation prevents datasets, model files and experiment outputs from consuming the smaller workspace volume.


In [15]:
from pathlib import Path

SOLUTION_ROOT = Path("/home/jovyan/chest-xray-ai-assistant")
DATA_ROOT = Path(
    "/home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data"
)

SOLUTION_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

root_paths = {
    "Solution root": SOLUTION_ROOT,
    "Data root": DATA_ROOT,
}

print("SOLUTION STORAGE INITIALIZATION")
print("-" * 100)

for name, path in root_paths.items():
    print(
        f"{name:<20} | "
        f"Exists: {path.exists()} | "
        f"Directory: {path.is_dir()} | "
        f"Path: {path}"
    )

SOLUTION STORAGE INITIALIZATION
----------------------------------------------------------------------------------------------------
Solution root        | Exists: True | Directory: True | Path: /home/jovyan/chest-xray-ai-assistant
Data root            | Exists: True | Directory: True | Path: /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data


## 1.3 Solution Workspace Structure

This section creates the primary workspace directories for notebooks, reusable source code, API services, user interface components, configuration, validation tests and solution documentation.


In [16]:
workspace_directories = [
    "notebooks",
    "src",
    "api",
    "ui",
    "configs",
    "tests",
    "reports",
]

created_workspace_paths = {}

for directory_name in workspace_directories:
    directory_path = SOLUTION_ROOT / directory_name
    directory_path.mkdir(parents=True, exist_ok=True)
    created_workspace_paths[directory_name] = directory_path

print("SOLUTION WORKSPACE STRUCTURE")
print("-" * 100)

for name, path in created_workspace_paths.items():
    print(
        f"{name:<15} | "
        f"Exists: {path.exists()} | "
        f"Path: {path}"
    )

SOLUTION WORKSPACE STRUCTURE
----------------------------------------------------------------------------------------------------
notebooks       | Exists: True | Path: /home/jovyan/chest-xray-ai-assistant/notebooks
src             | Exists: True | Path: /home/jovyan/chest-xray-ai-assistant/src
api             | Exists: True | Path: /home/jovyan/chest-xray-ai-assistant/api
ui              | Exists: True | Path: /home/jovyan/chest-xray-ai-assistant/ui
configs         | Exists: True | Path: /home/jovyan/chest-xray-ai-assistant/configs
tests           | Exists: True | Path: /home/jovyan/chest-xray-ai-assistant/tests
reports         | Exists: True | Path: /home/jovyan/chest-xray-ai-assistant/reports


## 1.4 Data and Artifact Storage Structure

This section creates dedicated storage locations for source data, processed data, trained models, checkpoints, external model caches, experiment tracking, explainability artifacts and generated outputs.


In [17]:
data_directories = [
    "raw",
    "processed",
    "models",
    "checkpoints",
    "hf-cache",
    "mlflow",
    "outputs",
    "explainability",
]

created_data_paths = {}

for directory_name in data_directories:
    directory_path = DATA_ROOT / directory_name
    directory_path.mkdir(parents=True, exist_ok=True)
    created_data_paths[directory_name] = directory_path

print("DATA AND ARTIFACT STORAGE STRUCTURE")
print("-" * 110)

for name, path in created_data_paths.items():
    print(
        f"{name:<15} | "
        f"Exists: {path.exists()} | "
        f"Path: {path}"
    )

DATA AND ARTIFACT STORAGE STRUCTURE
--------------------------------------------------------------------------------------------------------------
raw             | Exists: True | Path: /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/raw
processed       | Exists: True | Path: /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/processed
models          | Exists: True | Path: /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/models
checkpoints     | Exists: True | Path: /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/checkpoints
hf-cache        | Exists: True | Path: /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/hf-cache
mlflow          | Exists: True | Path: /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/mlflow
outputs         | Exists: True | Path: /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs
explainability  | Exists: True | Path: /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-d

# 2. Compute Resource Validation

## 2.1 CPU and Memory Allocation

This section validates the CPU and system-memory resources allocated to the Kubeflow runtime. Container-level limits are inspected because they represent the resources available to the solution more accurately than the underlying host specifications.


In [18]:
import os
from pathlib import Path

def read_cgroup_value(file_path):
    path = Path(file_path)
    return path.read_text().strip() if path.exists() else None


visible_cpu_count = os.cpu_count()
affinity_cpu_count = (
    len(os.sched_getaffinity(0))
    if hasattr(os, "sched_getaffinity")
    else visible_cpu_count
)

cpu_limit_raw = read_cgroup_value("/sys/fs/cgroup/cpu.max")
memory_limit_raw = read_cgroup_value("/sys/fs/cgroup/memory.max")
memory_used_raw = read_cgroup_value("/sys/fs/cgroup/memory.current")

if cpu_limit_raw and cpu_limit_raw != "max":
    quota, period = cpu_limit_raw.split()
    allocated_cpu = int(quota) / int(period)
else:
    allocated_cpu = affinity_cpu_count

if memory_limit_raw and memory_limit_raw != "max":
    memory_limit_gib = int(memory_limit_raw) / (1024 ** 3)
else:
    memory_limit_gib = None

if memory_used_raw:
    memory_used_gib = int(memory_used_raw) / (1024 ** 3)
else:
    memory_used_gib = None

print("CPU AND MEMORY ALLOCATION")
print("-" * 100)
print(f"{'Visible logical CPUs':<30}: {visible_cpu_count}")
print(f"{'CPU affinity count':<30}: {affinity_cpu_count}")
print(f"{'Container CPU allocation':<30}: {allocated_cpu:.2f}")

if memory_limit_gib is not None:
    print(f"{'Container memory limit':<30}: {memory_limit_gib:.2f} GiB")
else:
    print(f"{'Container memory limit':<30}: No explicit cgroup limit detected")

if memory_used_gib is not None:
    print(f"{'Current memory usage':<30}: {memory_used_gib:.2f} GiB")

CPU AND MEMORY ALLOCATION
----------------------------------------------------------------------------------------------------
Visible logical CPUs          : 256
CPU affinity count            : 256
Container CPU allocation      : 7.20
Container memory limit        : 18.00 GiB
Current memory usage          : 8.92 GiB


## 2.2 GPU Hardware Allocation

This section verifies the NVIDIA GPU assigned to the runtime and records its hardware capacity. GPU memory and compute capability determine the appropriate image resolution, batch size, mixed-precision strategy and language-model loading configuration.


In [19]:
import os
import subprocess

gpu_query = [
    "nvidia-smi",
    "--query-gpu=index,name,driver_version,memory.total,memory.used,memory.free,compute_cap",
    "--format=csv,noheader",
]

print("GPU HARDWARE ALLOCATION")
print("-" * 100)
print(f"{'CUDA_VISIBLE_DEVICES':<25}: {os.getenv('CUDA_VISIBLE_DEVICES', 'Not explicitly set')}")

try:
    gpu_result = subprocess.run(
        gpu_query,
        capture_output=True,
        text=True,
        check=False,
    )

    if gpu_result.returncode == 0:
        print(f"{'GPU details':<25}: {gpu_result.stdout.strip()}")
    else:
        print(f"{'GPU status':<25}: NVIDIA query failed")
        print(f"{'Message':<25}: {gpu_result.stderr.strip()}")
except FileNotFoundError:
    print(f"{'GPU status':<25}: nvidia-smi is not available")

GPU HARDWARE ALLOCATION
----------------------------------------------------------------------------------------------------
CUDA_VISIBLE_DEVICES     : Not explicitly set
GPU details              : 0, NVIDIA A100-SXM4-80GB, 535.309.01, 81920 MiB, 586 MiB, 80463 MiB, 8.0


## 2.3 PyTorch and CUDA Integration

This section verifies the installed PyTorch and TorchVision versions, CUDA integration, cuDNN support and Python-level access to the allocated NVIDIA GPU.


In [20]:
print("PYTORCH AND CUDA INTEGRATION")
print("-" * 100)

try:
    import torch
    import torchvision

    print(f"{'PyTorch version':<30}: {torch.__version__}")
    print(f"{'TorchVision version':<30}: {torchvision.__version__}")
    print(f"{'Compiled CUDA version':<30}: {torch.version.cuda}")
    print(f"{'CUDA available':<30}: {torch.cuda.is_available()}")
    print(f"{'cuDNN available':<30}: {torch.backends.cudnn.is_available()}")
    print(f"{'cuDNN version':<30}: {torch.backends.cudnn.version()}")

    if torch.cuda.is_available():
        device_properties = torch.cuda.get_device_properties(0)

        print(f"{'GPU accessible to PyTorch':<30}: {torch.cuda.get_device_name(0)}")
        print(
            f"{'GPU memory':<30}: "
            f"{device_properties.total_memory / (1024 ** 3):.2f} GiB"
        )
        print(
            f"{'Compute capability':<30}: "
            f"{device_properties.major}.{device_properties.minor}"
        )

except ImportError as error:
    print(f"{'Validation status':<30}: Required package is unavailable")
    print(f"{'Details':<30}: {error}")
except Exception as error:
    print(f"{'Validation status':<30}: Validation failed")
    print(f"{'Details':<30}: {type(error).__name__}: {error}")

PYTORCH AND CUDA INTEGRATION
----------------------------------------------------------------------------------------------------
PyTorch version               : 2.5.1+cu121
TorchVision version           : 0.20.1+cu121
Compiled CUDA version         : 12.1
CUDA available                : True
cuDNN available               : True
cuDNN version                 : 90100
GPU accessible to PyTorch     : NVIDIA A100-SXM4-80GB
GPU memory                    : 79.15 GiB
Compute capability            : 8.0


# 3. Software Dependency Validation

## 3.1 Solution Package Inventory

This section inventories the Python packages required by the solution. The validation covers data engineering, computer vision, explainability, language-model integration, API services, user interface development, experiment tracking and automated testing.


In [21]:
from importlib.metadata import PackageNotFoundError, version

package_groups = {
    "Data and evaluation": [
        "numpy",
        "pandas",
        "scipy",
        "scikit-learn",
        "Pillow",
        "matplotlib",
        "seaborn",
    ],
    "Medical imaging and explainability": [
        "medmnist",
        "captum",
    ],
    "Language model integration": [
        "transformers",
        "datasets",
        "accelerate",
        "bitsandbytes",
        "sentencepiece",
    ],
    "API and user interface": [
        "fastapi",
        "uvicorn",
        "python-multipart",
        "streamlit",
        "httpx",
    ],
    "Operations and validation": [
        "mlflow",
        "PyYAML",
        "pytest",
    ],
}

missing_packages = []

print("SOLUTION PACKAGE INVENTORY")
print("-" * 100)

for group_name, packages in package_groups.items():
    print(f"\n{group_name}")

    for package_name in packages:
        try:
            installed_version = version(package_name)
            status = f"Installed ({installed_version})"
        except PackageNotFoundError:
            status = "Missing"
            missing_packages.append(package_name)

        print(f"  {package_name:<22}: {status}")

print("\n" + "-" * 100)
print(f"Installed packages : {sum(len(v) for v in package_groups.values()) - len(missing_packages)}")
print(f"Missing packages   : {len(missing_packages)}")
print(f"Missing list       : {', '.join(missing_packages) if missing_packages else 'None'}")

SOLUTION PACKAGE INVENTORY
----------------------------------------------------------------------------------------------------

Data and evaluation
  numpy                 : Installed (1.26.4)
  pandas                : Installed (2.2.3)
  scipy                 : Installed (1.15.1)
  scikit-learn          : Installed (1.6.1)
  Pillow                : Installed (11.2.1)
  matplotlib            : Installed (3.10.0)
  seaborn               : Installed (0.13.2)

Medical imaging and explainability
  medmnist              : Installed (3.0.2)
  captum                : Installed (0.7.0)

Language model integration
  transformers          : Installed (4.48.3)
  datasets              : Installed (3.2.0)
  accelerate            : Installed (1.3.0)
  bitsandbytes          : Installed (0.45.2)
  sentencepiece         : Installed (0.2.0)

API and user interface
  fastapi               : Installed (0.115.12)
  uvicorn               : Installed (0.34.0)
  python-multipart      : Installed (0.0.20)
  s

## 3.3 Dependency Import Validation

This section performs an import-level compatibility test across the solution’s principal libraries. Successful package installation does not always guarantee runtime compatibility, particularly for GPU, scientific-computing and compiled dependencies.


In [22]:
from importlib import import_module

module_groups = {
    "Computer vision": [
        "torch",
        "torchvision",
        "medmnist",
        "captum",
    ],
    "Language model": [
        "transformers",
        "datasets",
        "accelerate",
        "bitsandbytes",
        "sentencepiece",
    ],
    "Service layer": [
        "fastapi",
        "uvicorn",
        "multipart",
        "streamlit",
    ],
    "Operations": [
        "mlflow",
        "yaml",
        "pytest",
    ],
}

import_failures = []

print("DEPENDENCY IMPORT VALIDATION")
print("-" * 100)

for group_name, module_names in module_groups.items():
    print(f"\n{group_name}")

    for module_name in module_names:
        try:
            import_module(module_name)
            status = "Ready"
        except Exception as error:
            status = f"Failed — {type(error).__name__}: {error}"
            import_failures.append(module_name)

        print(f"  {module_name:<20}: {status}")

print("\n" + "-" * 100)
print(f"Validated modules : {sum(len(v) for v in module_groups.values())}")
print(f"Import failures   : {len(import_failures)}")
print(
    f"Failure list      : "
    f"{', '.join(import_failures) if import_failures else 'None'}"
)

DEPENDENCY IMPORT VALIDATION
----------------------------------------------------------------------------------------------------

Computer vision
  torch               : Ready
  torchvision         : Ready
  medmnist            : Ready
  captum              : Ready

Language model
  transformers        : Ready
  datasets            : Ready
  accelerate          : Ready
  bitsandbytes        : Ready
  sentencepiece       : Ready

Service layer
  fastapi             : Ready
  uvicorn             : Ready
  multipart           : Ready
  streamlit           : Ready

Operations


/opt/conda/lib/python3.11/site-packages/mlflow/utils/requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


  mlflow              : Ready
  yaml                : Ready
  pytest              : Ready

----------------------------------------------------------------------------------------------------
Validated modules : 16
Import failures   : 0
Failure list      : None


## 3.4 Kubeflow Dependency Compatibility

This section verifies the compatibility of Protobuf with the Kubeflow Pipelines SDK. Maintaining this compatibility protects future pipeline orchestration and experiment automation from dependency-related failures.


In [23]:
from importlib.metadata import version
from packaging.version import Version

import kfp
import google.protobuf

protobuf_version = version("protobuf")
kfp_version = version("kfp")
pipeline_spec_version = version("kfp-pipeline-spec")

protobuf_compatible = (
    Version("4.21.1")
    <= Version(protobuf_version)
    < Version("5.0.0")
)

print("KUBEFLOW DEPENDENCY COMPATIBILITY")
print("-" * 100)
print(f"{'Protobuf version':<30}: {protobuf_version}")
print(f"{'Kubeflow Pipelines SDK':<30}: {kfp_version}")
print(f"{'KFP pipeline specification':<30}: {pipeline_spec_version}")
print(f"{'Protobuf compatibility':<30}: {protobuf_compatible}")

KUBEFLOW DEPENDENCY COMPATIBILITY
----------------------------------------------------------------------------------------------------
Protobuf version              : 4.25.3
Kubeflow Pipelines SDK        : 2.11.0
KFP pipeline specification    : 0.6.0
Protobuf compatibility        : True


# 4. Functional Runtime Validation

## 4.1 GPU Tensor and Mixed-Precision Test

This section performs a controlled tensor operation on the allocated GPU. The test confirms functional CUDA execution and identifies the mixed-precision formats supported by the hardware.


In [24]:
import time
import torch

device = torch.device("cuda:0")

matrix_size = 2048
left_matrix = torch.randn(
    matrix_size,
    matrix_size,
    device=device,
    dtype=torch.float16,
)
right_matrix = torch.randn(
    matrix_size,
    matrix_size,
    device=device,
    dtype=torch.float16,
)

torch.cuda.synchronize()
start_time = time.perf_counter()

result_matrix = torch.matmul(left_matrix, right_matrix)

torch.cuda.synchronize()
elapsed_ms = (time.perf_counter() - start_time) * 1000

print("GPU FUNCTIONAL VALIDATION")
print("-" * 100)
print(f"{'Execution device':<30}: {result_matrix.device}")
print(f"{'Tensor precision':<30}: {result_matrix.dtype}")
print(f"{'Output shape':<30}: {tuple(result_matrix.shape)}")
print(f"{'Finite output':<30}: {torch.isfinite(result_matrix).all().item()}")
print(f"{'FP16 supported':<30}: True")
print(f"{'BF16 supported':<30}: {torch.cuda.is_bf16_supported()}")
print(f"{'Execution time':<30}: {elapsed_ms:.2f} ms")

del left_matrix, right_matrix, result_matrix
torch.cuda.empty_cache()

GPU FUNCTIONAL VALIDATION
----------------------------------------------------------------------------------------------------
Execution device              : cuda:0
Tensor precision              : torch.float16
Output shape                  : (2048, 2048)
Finite output                 : True
FP16 supported                : True
BF16 supported                : True
Execution time                : 91.22 ms


# 5. Centralized Path Configuration

## 5.1 Solution Path Registry

This section creates a centralized path registry for notebooks, services, datasets, models, experiment artifacts and runtime caches. Centralized configuration ensures consistent storage usage across training, evaluation, API and monitoring components.


In [25]:
import yaml

runtime_cache_paths = {
    "huggingface": DATA_ROOT / "hf-cache",
    "torch": DATA_ROOT / "models" / "torch-cache",
    "pip": DATA_ROOT / "pip-cache",
}

for cache_path in runtime_cache_paths.values():
    cache_path.mkdir(parents=True, exist_ok=True)

path_configuration = {
    "solution": {
        "root": str(SOLUTION_ROOT),
        "notebooks": str(SOLUTION_ROOT / "notebooks"),
        "source": str(SOLUTION_ROOT / "src"),
        "api": str(SOLUTION_ROOT / "api"),
        "ui": str(SOLUTION_ROOT / "ui"),
        "configs": str(SOLUTION_ROOT / "configs"),
        "tests": str(SOLUTION_ROOT / "tests"),
        "reports": str(SOLUTION_ROOT / "reports"),
    },
    "data": {
        "root": str(DATA_ROOT),
        "raw": str(DATA_ROOT / "raw"),
        "processed": str(DATA_ROOT / "processed"),
        "models": str(DATA_ROOT / "models"),
        "checkpoints": str(DATA_ROOT / "checkpoints"),
        "mlflow": str(DATA_ROOT / "mlflow"),
        "outputs": str(DATA_ROOT / "outputs"),
        "explainability": str(DATA_ROOT / "explainability"),
    },
    "cache": {
        name: str(path)
        for name, path in runtime_cache_paths.items()
    },
}

path_config_file = SOLUTION_ROOT / "configs" / "paths.yaml"

with path_config_file.open("w", encoding="utf-8") as file:
    yaml.safe_dump(
        path_configuration,
        file,
        sort_keys=False,
        default_flow_style=False,
    )

print("SOLUTION PATH REGISTRY")
print("-" * 100)
print(f"Configuration file : {path_config_file}")
print(f"File created       : {path_config_file.exists()}")
print()
print(path_config_file.read_text(encoding="utf-8"))

SOLUTION PATH REGISTRY
----------------------------------------------------------------------------------------------------
Configuration file : /home/jovyan/chest-xray-ai-assistant/configs/paths.yaml
File created       : True

solution:
  root: /home/jovyan/chest-xray-ai-assistant
  notebooks: /home/jovyan/chest-xray-ai-assistant/notebooks
  source: /home/jovyan/chest-xray-ai-assistant/src
  api: /home/jovyan/chest-xray-ai-assistant/api
  ui: /home/jovyan/chest-xray-ai-assistant/ui
  configs: /home/jovyan/chest-xray-ai-assistant/configs
  tests: /home/jovyan/chest-xray-ai-assistant/tests
  reports: /home/jovyan/chest-xray-ai-assistant/reports
data:
  root: /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data
  raw: /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/raw
  processed: /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/processed
  models: /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/models
  checkpoints: /home/jovyan/apicdsa2-da

## 5.2 Runtime Cache and Experiment Tracking Configuration

This section configures runtime cache locations and the local MLflow tracking store. The configuration ensures that large model downloads, dataset caches and experiment artifacts remain on the dedicated data volume.


In [26]:
import os
import mlflow

runtime_environment = {
    "HF_HOME": path_configuration["cache"]["huggingface"],
    "HF_DATASETS_CACHE": str(
        Path(path_configuration["cache"]["huggingface"]) / "datasets"
    ),
    "TORCH_HOME": path_configuration["cache"]["torch"],
    "PIP_CACHE_DIR": path_configuration["cache"]["pip"],
    "MLFLOW_TRACKING_URI": Path(
        path_configuration["data"]["mlflow"]
    ).as_uri(),
    "TOKENIZERS_PARALLELISM": "false",
}

for variable_name, variable_value in runtime_environment.items():
    os.environ[variable_name] = variable_value

Path(runtime_environment["HF_DATASETS_CACHE"]).mkdir(
    parents=True,
    exist_ok=True,
)

mlflow.set_tracking_uri(runtime_environment["MLFLOW_TRACKING_URI"])

print("RUNTIME CONFIGURATION")
print("-" * 110)

for variable_name, variable_value in runtime_environment.items():
    print(f"{variable_name:<25}: {variable_value}")

print()
print(f"{'Active MLflow URI':<25}: {mlflow.get_tracking_uri()}")

RUNTIME CONFIGURATION
--------------------------------------------------------------------------------------------------------------
HF_HOME                  : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/hf-cache
HF_DATASETS_CACHE        : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/hf-cache/datasets
TORCH_HOME               : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/models/torch-cache
PIP_CACHE_DIR            : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/pip-cache
MLFLOW_TRACKING_URI      : file:///home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/mlflow
TOKENIZERS_PARALLELISM   : false

Active MLflow URI        : file:///home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/mlflow


# 6. Environment Readiness Assessment

## 6.1 Readiness Gate

This section consolidates the platform, storage, dependency and GPU validations into a final readiness decision. The solution proceeds to data engineering only when all mandatory conditions are satisfied.


In [27]:
import os
import shutil
import sys

workspace_free_gib = shutil.disk_usage(SOLUTION_ROOT).free / (1024 ** 3)
data_free_gib = shutil.disk_usage(DATA_ROOT).free / (1024 ** 3)
gpu_memory_gib = (
    torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if torch.cuda.is_available()
    else 0
)

readiness_checks = {
    "Supported Python runtime": sys.version_info[:2] == (3, 11),
    "Solution workspace available": SOLUTION_ROOT.exists(),
    "Solution workspace writable": os.access(SOLUTION_ROOT, os.W_OK),
    "Dedicated data volume available": DATA_ROOT.exists(),
    "Dedicated data volume writable": os.access(DATA_ROOT, os.W_OK),
    "Minimum data storage available": data_free_gib >= 15,
    "PyTorch CUDA integration": torch.cuda.is_available(),
    "Minimum GPU memory available": gpu_memory_gib >= 16,
    "Required package imports": len(import_failures) == 0,
    "Kubeflow Protobuf compatibility": protobuf_compatible,
    "MLflow tracking configured": (
        mlflow.get_tracking_uri()
        == runtime_environment["MLFLOW_TRACKING_URI"]
    ),
}

failed_checks = [
    check_name
    for check_name, passed in readiness_checks.items()
    if not passed
]

print("ENVIRONMENT READINESS ASSESSMENT")
print("-" * 100)

for check_name, passed in readiness_checks.items():
    print(f"{check_name:<42}: {'PASS' if passed else 'FAIL'}")

print("\nRESOURCE SUMMARY")
print("-" * 100)
print(f"{'Workspace free space':<42}: {workspace_free_gib:.2f} GiB")
print(f"{'Data-volume free space':<42}: {data_free_gib:.2f} GiB")
print(f"{'Available GPU memory':<42}: {gpu_memory_gib:.2f} GiB")

print("\nFINAL STATUS")
print("-" * 100)
print(
    "READY FOR DATA ENGINEERING"
    if not failed_checks
    else f"NOT READY — Failed checks: {', '.join(failed_checks)}"
)

ENVIRONMENT READINESS ASSESSMENT
----------------------------------------------------------------------------------------------------
Supported Python runtime                  : PASS
Solution workspace available              : PASS
Solution workspace writable               : PASS
Dedicated data volume available           : PASS
Dedicated data volume writable            : PASS
Minimum data storage available            : PASS
PyTorch CUDA integration                  : PASS
Minimum GPU memory available              : PASS
Required package imports                  : PASS
Kubeflow Protobuf compatibility           : PASS
MLflow tracking configured                : PASS

RESOURCE SUMMARY
----------------------------------------------------------------------------------------------------
Workspace free space                      : 7.76 GiB
Data-volume free space                    : 20.67 GiB
Available GPU memory                      : 79.15 GiB

FINAL STATUS
---------------------------------

## Readiness Outcome

The runtime satisfies all infrastructure requirements for the Chest X-Ray Analysis and Explanation Assistant.

The validated environment provides a Python 3.11 runtime, container-governed CPU and memory resources, an NVIDIA A100 GPU with approximately 80 GiB VRAM, CUDA-enabled PyTorch, all required solution dependencies, compatible Kubeflow packages and dedicated artifact storage.

The platform is approved for ChestMNIST data acquisition, multilabel exploratory analysis and preprocessing.
